# Employee Data Cleaning and Visualization

This notebook cleans the raw employee CSV, validates the result, exports `cleaned_data.csv`, and creates four charts from the cleaned data.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = Path('sample_data_cleaning_project - Sample_data_cleaning_project.csv')
OUTPUT_FILE = Path('cleaned_data.csv')
PLOT_DIR = Path('visualizations')
PLOT_DIR.mkdir(exist_ok=True)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Dataset not found: {INPUT_FILE}')
data = pd.read_csv(INPUT_FILE)
if data.empty:
    raise ValueError('The dataset is empty.')
print('Raw shape:', data.shape)

In [ ]:
# Standardize text and convert fields before cleaning.
for column in data.select_dtypes(include='object').columns:
    data[column] = data[column].astype('string').str.strip()
data['Name'] = data['Name'].str.lower()
data['Department'] = data['Department'].str.lower()
data['Age'] = pd.to_numeric(data['Age'], errors='coerce')
data['Salary'] = pd.to_numeric(data['Salary'], errors='coerce')
for column in ['Age', 'Salary']:
    data[column] = data[column].fillna(data[column].median())
data['Join_Date'] = pd.to_datetime(data['Join_Date'], errors='coerce')
data = data.dropna(subset=['Join_Date']).drop_duplicates().copy()

# Remove salary outliers using the standard IQR rule.
q1, q3 = data['Salary'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
data = data[data['Salary'].between(lower, upper)].copy()
data['Age'] = data['Age'].round().astype(int)
data = pd.get_dummies(data, columns=['Department'], prefix='Department', dtype=int)
department_columns = [c for c in data.columns if c.startswith('Department_')]
if data.isna().sum().sum() or data.duplicated().any() or data.empty:
    raise ValueError('Validation failed.')
data.to_csv(OUTPUT_FILE, index=False)
print('Cleaned shape:', data.shape)

In [ ]:
# Reconstruct readable labels for charts.
chart_data = pd.read_csv(OUTPUT_FILE, parse_dates=['Join_Date'])
department_columns = [c for c in chart_data.columns if c.startswith('Department_')]
def department_label(row):
    for column in department_columns:
        if row[column] == 1:
            return column.replace('Department_', '').upper()
    return 'Unknown'
chart_data['Department_Label'] = chart_data.apply(department_label, axis=1)
chart_data['Join_Year'] = chart_data['Join_Date'].dt.year

def save_chart(name):
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'{name}.svg', bbox_inches='tight')
    plt.savefig(PLOT_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()

# 1. Department counts
counts = chart_data['Department_Label'].value_counts().reindex(['HR', 'IT', 'Unknown'], fill_value=0)
plt.figure(figsize=(7, 4)); plt.bar(counts.index, counts.values, color=['#4C78A8', '#F58518', '#B0B0B0']); plt.title('Employees by Department'); plt.xlabel('Department'); plt.ylabel('Employees'); save_chart('bar_department_counts')

# 2. Age and salary relationship
plt.figure(figsize=(7, 4)); plt.scatter(chart_data['Age'], chart_data['Salary'], c='#54A24B', edgecolors='black', alpha=.8); plt.title('Age vs Salary'); plt.xlabel('Age'); plt.ylabel('Salary'); plt.grid(alpha=.25); save_chart('scatter_age_salary')

# 3. Average salary by joining year
average_salary = chart_data.groupby('Join_Year')['Salary'].mean()
plt.figure(figsize=(7, 4)); plt.plot(average_salary.index, average_salary.values, marker='o', color='#E45756'); plt.title('Average Salary by Joining Year'); plt.xlabel('Joining year'); plt.ylabel('Average salary'); plt.grid(alpha=.25); save_chart('line_average_salary_by_year')

# 4. Age ranges
bins = [20, 29, 39, 49, 59]; labels = ['21-29', '30-39', '40-49', '50-59']
age_groups = pd.cut(chart_data['Age'], bins=bins, labels=labels, include_lowest=True).value_counts().sort_index()
plt.figure(figsize=(7, 4)); plt.bar(age_groups.index.astype(str), age_groups.values, color='#72B7B2'); plt.title('Employee Age Distribution'); plt.xlabel('Age range'); plt.ylabel('Employees'); save_chart('histogram_age_distribution')
print('Charts saved to', PLOT_DIR.resolve())